In [18]:
#import libraries
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
#this is for making custom stop words
from sklearn.feature_extraction._stop_words import ENGLISH_STOP_WORDS

In [2]:
#read csv files, give their columns titles
drawings = pd.read_csv(r"C:\Users\gramos1\OneDrive - Reworld\DrawingNames.csv",sep=',',header=0,names=['Drawing Title'])
labels = pd.read_csv(r"C:\Users\gramos1\OneDrive - Reworld\drawinglabels.csv", sep=',',header=0, names=['Categories','Keywords'])

In [3]:
#just testing the next two cells
drawings.head(5)

,Drawing Title
0,BIC-0200 - INSTRMNTN & CONTROL DIAGS. SYMB. & ...
1,BIC-0201 - MAIN & DUMP STM SYS. INSTRMNTN & CT...
2,BIC-0202 - EXT. STM & AUX. STM SYS. INSTRN & C...
3,BIC-0203 - FEEDWATER SYS. INSTRMTN & CTL DIAGS...
4,BIC-0204 - CONDS & MKE - UP WATER SYS. INSTR &...


In [25]:
labels.head(5)

,Categories,Keywords
0,Generator,generator ac dc voltage adjuster regulator
1,Electrical Distribution,transformer kv switchyard motor electrical ele...
2,Condensate System,condensate desuperhtr deaerator condenser
3,Feed Water,feedwater superheater feed pump drum
4,Water Treatment and Sampling,treatment sampling


In [4]:
#heres my attempt to remove any alphanumerical codes, keeping them can mess up training
drawings['Drawing Title'] = drawings['Drawing Title'].apply(lambda x: x.split('-', 1)[1] if "-" in x else x)

In [5]:
drawings.head(5)

,Drawing Title
0,0200 - INSTRMNTN & CONTROL DIAGS. SYMB. & LEGE...
1,0201 - MAIN & DUMP STM SYS. INSTRMNTN & CTL DI...
2,0202 - EXT. STM & AUX. STM SYS. INSTRN & CTL D...
3,0203 - FEEDWATER SYS. INSTRMTN & CTL DIAGS IND...
4,0204 - CONDS & MKE - UP WATER SYS. INSTR & CTL...


In [56]:
custom_stop_words = list(ENGLISH_STOP_WORDS) + ['sys',
                                                'installation',
                                                'detail',
                                               'schem',
                                               'diagram',
                                               'dwg',
                                               'diag',
                                               'plans',
                                               'layout',
                                               'details',
                                               'arrgt',]

In [57]:


#vect to bring in tfidf vectorizer
#stop words is to filter noise, english here to filter out generic english

vect = TfidfVectorizer(
     #trying to get rid of english and numbers
                       stop_words= custom_stop_words,
    #filter out anything not alphabetic
                        token_pattern=r'(?u)\b[a-zA-Z]+\b',
     #put this 1-2 so i can get bigrams, might need to increase? 1-3?
                      ngram_range=(1,3),
     #I want features to be words i think
                    analyzer = 'word',
      #min_df to try and filter out irrelevant words? usually drawings are in batches so if it's
        #Not important enough to be in a batch then dont count it
                     min_df=3,)

#fit_transform applied to drawing title names,
#Learn vocabulary and idf, return document-term matrix.
x = vect.fit_transform(drawings['Drawing Title'])

In [58]:
#output of feature names for transformation
#(texts turns into numerical features)
print(vect.get_feature_names_out()[:200])

['ab' 'ab elev' 'absorber' 'absorber cone' 'absorber cone sub'
 'absorber scraper' 'ac' 'ac current' 'ac dc' 'ac power' 'access'
 'access collector' 'access collector plan' 'access door'
 'access door ass' 'access door box' 'access road' 'access road sh' 'acid'
 'acid tank' 'actuator' 'add' 'add conden' 'add conden supp'
 'add condenser' 'add condenser support' 'admin' 'admin bldg'
 'admin bldg el' 'admin bldg fl' 'admin bldgs' 'admin bldgs fdn' 'aggreg'
 'ahu' 'ahu control' 'ahu control sh' 'air' 'air boiler' 'air boiler b'
 'air compressors' 'air conn' 'air cooled' 'air cooled condenser'
 'air cooled steam' 'air duct' 'air fan' 'air fan sh' 'air flow'
 'air flow alarms' 'air flue' 'air flue gas' 'air handling'
 'air handling units' 'air intake' 'air piping' 'air piping boiler'
 'air piping turb' 'air press' 'air press duct' 'air prim' 'air prim air'
 'air purge' 'air sealing' 'air sealing plates' 'air secdry' 'alarm'
 'alarms' 'alarms lm' 'alarms lm trmt' 'alarms rxp' 'alarms rxp sh'

In [59]:
#prints out matrix with feature values, many will be zero, presented in this way to save memory
#remember matrix is one row per document and one col per token
#value = tfidf score 0-1, and the score quantifies how important that word is to the title
print(x[:50])


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 441 stored elements and shape (50, 3466)>
  Coords	Values
  (0, 1687)	0.46517022132800967
  (0, 711)	0.249961846165101
  (0, 893)	0.354120106409519
  (0, 1790)	0.40094749674288294
  (0, 1623)	0.41459827291207735
  (0, 2778)	0.16470352506940833
  (0, 1625)	0.4857538642888444
  (1, 1687)	0.3210711841245743
  (1, 893)	0.24442184102548517
  (1, 1623)	0.28616526234171624
  (1, 2778)	0.11368216063477889
  (1, 1625)	0.33527848785129444
  (1, 1915)	0.2708418329119518
  (1, 986)	0.31543291491366104
  (1, 2937)	0.2746830523139067
  (1, 794)	0.2289705236719368
  (1, 1688)	0.327580088246883
  (1, 797)	0.3210711841245743
  (1, 1689)	0.34470056748560585
  (2, 893)	0.260911397970108
  (2, 2778)	0.12135155897295559
  (2, 2937)	0.5864282739810073
  (2, 794)	0.24441768041082593
  (2, 797)	0.3427317753045075
  (2, 1160)	0.20140596808250635
  :	:
  (47, 262)	0.2901634651595562
  (48, 397)	0.2652734610885768
  (48, 255)	0.2620063139195348
  (48,